## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import httpx
from langchain_groq import ChatGroq

# Disable SSL verification to work around corporate proxy/firewall
client = httpx.Client(verify=False)
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, http_client=client)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The capital of The Moon is Lunaria. Located on the Moon's south pole, Lunaria is a marvel of human ingenuity and technological advancement. The city is nestled within a massive, dome-shaped structure that maintains a stable atmosphere and protects its inhabitants from the harsh lunar environment.

### Geography and Climate

Lunaria is situated on the southern edge of the Moon's south pole, near the crater Shackleton. The city's unique location allows for optimal solar energy harvesting and provides a breathtaking view of the Earth from the lunar horizon. The city's climate is controlled within the dome, maintaining a comfortable temperature range of 20-25°C (68-77°F) and a stable atmospheric pressure.

### History

Lunaria was founded in 2055 by a coalition of governments and private corporations, marking a significant milestone in human space exploration and settlement. The city was designed to serve as a hub for scientific research, lunar resource extraction, and interplanetary trade

## Image input

In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'Screenshot 2026-03-06 at 3.48.01\u202fAM.png', 'type': 'image/png', 'size': 4452493, 'content': <memory at 0x1152641c0>, 'last_modified': datetime.datetime(2026, 3, 5, 22, 18, 7, 284000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [15]:
# Create a new agent with a neutral system prompt for image analysis
image_agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant that can analyze images and text.",
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about content of this image."},
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
])

response = image_agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The image presents a screenshot of a video player, showcasing a list of finalists for the "Software Product Engineering Rising Star" award. The title, in large white text, is prominently displayed at the top center of the screen.

**Title and Finalists:**

* **Title:** "Software Product Engineering Rising Star Finalists"
* **Finalists:**
	+ Syed Ajaz
	+ Shantam Bhuraria
	+ Tenneti Mownika

**Video Player Interface:**

* **Logo:** A logo for "TNN TECHY NEWS NETWORK" is situated in the bottom-left corner.
* **Video Controls:** The video player interface features various controls, including:
	+ A progress bar
	+ Play/pause button
	+ Volume control
	+ Full-screen button
* **Video Information:**
	+ Video stream: 2026 Techy Awards
	+ Time elapsed: 1:33:29/1:44:15

**Background:**

* The background of the image features a dark blue stage with spotlights shining down, creating a sense of excitement and anticipation. The overall atmosphere suggests that the image is from a live event or a recor

## Audio input

In [23]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 100  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 1000/1000 [01:44<00:00,  9.59it/s]


Done.


In [24]:
from groq import Groq

# Step 1: Transcribe audio using Groq's Whisper model
groq_client = Groq(http_client=client)

transcription = groq_client.audio.transcriptions.create(
    file=("recording.wav", wav_bytes),
    model="whisper-large-v3",
)

print("Transcription:", transcription.text)

# Step 2: Send transcription to the LLM for analysis
audio_agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant.",
)

audio_question = HumanMessage(content=[
    {"type": "text", "text": f"The user recorded an audio message. Here is the transcription:\n\n\"{transcription.text}\"\n\nTell me about this audio.and respond to the user."}
])

response = audio_agent.invoke(
    {"messages": [audio_question]}
)

print(response['messages'][-1].content)

Transcription:  హాయ్ గీవు మీ సాపీ సివో కంసోటంట్ సినేర్ లవెల్ ఇంట్రవి కొస్యంస్ కొసించు కిల్లాక్ సలరి పెక్కేచ్ సివా కంసల్టెంట్ రోల్ మిక్స్తు మిక్స్తు మిల్లిక్ మిల్లిక్ మిల్లిక్ మిల్లిక్కులు పాలుకులు పాలుకులు
It seems like the audio message you provided is in Telugu, and it's a bit unclear due to possible misinterpretation or background noise. However, I can try to make out some parts of it.

From what I can understand, the message seems to be:

"Hi, I'm your SAP FI/CO consultant. I'm at a senior level with extensive experience. I'm looking for a challenging role as a consultant. I'm available for immediate joining. Please call me."

Please let me know if I'm correct or not. If I'm close, I'd be happy to help you respond to the user.

Here's a potential response:

"Hello! Thank you for reaching out. I'd be happy to discuss the consultant role further. Can you please share your resume and availability for a call?" 

If I'm not correct, please provide more context or clarify the message, an